# Week 7: Planning and Learning Integration

Craig Rudman  
Regis University  
MSDS684 Reinforcement Learning  
Prof. Mike Busch  

## Section 1: Project Overview 

This lab examines how model-learning changes the sample efficiency of tabular Q-learning methods. The lab also investigates how to detect when a model needs updating due to changes in the environment, and how stale entries are overwritten in such cases. The lab will help explain under what conditions a model adds value and what mechanism delivers it.

I explore the concept of *planning*, which Sutton and Barto define as “any computational process that takes a model as input and produces or improves a policy for interacting with the modeled environment” (Sutton and Barto, p. 160). I will demonstrate how planning and Q-learning are integrated with the Dyna architecture, and compare the ability of Dyna-Q and Dyna-Q+ to respond to environment changes. I will look at prioritized sweeping to improve planning efficiency. Finally, I will examine the computational trade-offs involved when using model-based versus model-free methods.

The lab uses Gymnasium's Taxi-v4 in deterministic mode to conduct this investigation, since Taxi-v3 has been deprecated. This release has the following attributes:

- 500 discrete states (25 taxi positions in a 5x5 grid, 5 possible passenger locations, and 4 destination locations)
- 6 discrete actions (up, down, left, right, pickup, dropoff)
- Rewards per step: +20 for delivering a passenger, -10 for incorrect pickup or dropoff, and -1 for all other steps
- Episodes terminate when a taxi drops off a passenger and terminate after 200 steps when the TimeLimit wrapper is used to construct the environment

I have the following expectations from this lab:

- Dyna-Q will accumulate rewards faster, require fewer steps per episode, and fewer episodes overall than Q-learning alone to achieve optimal performance. Q-learning alone makes one policy update per step; with planning, multiple Q-updates are made per real step depending on the number of planning steps configured. Sutton and Barto demonstrate this on the maze task: without planning, the policy extends backward one step per episode; with planning, it extends much further per episode (Sutton and Barto, p. 165).

- Dyna-Q+ will adapt to environment changes in fewer steps than Dyna-Q. Dyna-Q+ adds an exploration bonus based on the last time a state-action pair was tried in the environment, increasing the chance that stale transitions will be revisited.

- Prioritized sweeping will converge on an optimal solution with fewer updates than uniform random planning. It maintains a reverse index of the model that maps states to their state-action predecessors, and a priority queue ranked by absolute TD error. These mechanisms direct the planning loop to propagate updates backwards to state-action predecessors whose Q-values are most likely to change, rather than sampling them uniformly.

## Section 2: Deliverables

### GitHub Repository

GitHub Repository: https://github.com/craig-rudman/MSDS684/tree/main/W7

### Implementation Summary

This lab implements four tabular reinforcement learning agents in a class hierarchy designed for clear separation of concerns. A base Q-learning agent performs one-step TD updates on a NumPy Q-table using real environment interactions. Two Dyna-Q variants extend this baseline by maintaining a deterministic model dictionary mapping (s, a) → (r, s'), which supports simulated planning steps between real steps. The first uses uniform random sampling from the model; the second uses prioritized sweeping to replace random sampling with a priority queue ranked by absolute TD error, backed by a reverse lookup table that propagates value updates to predecessor states. A Dyna-Q+ variant adds a staleness bonus for each state-action pair to accelerate recovery when changes happen in the environment, demonstrated via a custom Gymnasium wrapper that swaps pickup and dropoff locations at step 1,000. Each agent subclasses only what it extends, making algorithmic differences explicit.

#### Key Results and Analysis

**Dyna-Q vs. Q-learning**

Planning steps compress the real experience required to reach competence. In the n-planning sweep, Dyna-Q (n=10) reached 90% termination rate in 14,247 real steps versus 51,903 for Q-learning (a 3.6x improvement), while Dyna-Q (n=50) reached it in 9,360 steps (a 5.5x improvement). On the episode axis, the difference is equally stark: Q-learning required roughly 375 episodes to converge while Dyna-Q (n=50) needed only around 75. Each planning step replays a previously observed transition at zero real environment cost, compressing the effective learning timeline. This confirms the prediction in Sutton and Barto: "Indirect methods often make fuller use of a limited amount of experience and thus achieve a better policy with fewer environmental interactions" (p. 162).

![](../img/v2/cumulative_reward.png)

*Figure 1. Cumulative reward over real steps for Q-learning (n=0) and Dyna-Q (n=5, 10, 50). More planning steps shifts the inflection point leftward, meaning the agent accumulates positive reward sooner.*

![](../img/v2/episodes_to_optimal.png)

*Figure 2. Trailing-50 termination rate over episodes. Dyna-Q (n=50) crosses the 90% threshold around episode 75; Q-learning requires roughly 375.*

![](../img/v2/sample_efficiency.png)

*Figure 3. Trailing-50 termination rate over real environment steps. Dyna-Q (n=50) reaches 90% termination in about 9,500 steps, versus around 55,000 for Q-learning.*

**Dynamic environment and Dyna-Q+**

When the pickup and dropoff locations swapped at step 1,000, Dyna-Q's stale model actively reinforced wrong Q-values, slowing recovery. This illustrates model bias: planning errors compound in ways that model-free updates do not. Dyna-Q+ addressed this with a staleness bonus that grows with the time since each state-action pair was last visited, eventually overriding stale model values and driving re-exploration. The result is faster recovery after the environmental change.

![](../img/v3/termination_rate.png)

*Figure 4. Trailing-50 termination rate over real steps on the dynamic environment (pickup and dropoff locations swapped at step 1,000). Dyna-Q+ (κ=0.0001) leads Dyna-Q (n=10) consistently from around step 5,000 onward, reflecting faster re-exploration and correction of stale model entries after the environmental change.*

**Prioritized sweeping**

Directing the planning budget toward high-TD-error transitions achieves the same sample efficiency as Dyna-Q (n=50) at a fraction of the compute cost. Prioritized sweeping reached 90% termination in 9,100 real steps and approximately 80 episodes, versus 14,247 steps and 119 episodes for Dyna-Q (n=10), using the same planning budget per step. The wall-time comparison is the most striking result: Dyna-Q (n=50) required 37.9s per seed to reach similar sample efficiency; prioritized sweeping required 5.6s.

![](../img/v4/heatmap.png)

*Figure 5. Normalized trade-off scores across sample efficiency and wall time. Prioritized sweeping matches Dyna-Q (n=50) on sample efficiency while requiring one-seventh the wall time.*

**Synthesis**

Model-based planning adds value when real environment interactions are expensive and the model is accurate. In Taxi-v4's deterministic environment, simulated experience is a near-lossless substitute for real experience, which is why planning agents converge 3 to 6 times faster in real steps than Q-learning. The cost is compute: wall time scales linearly with the planning budget under uniform random sampling. Prioritized sweeping breaks that scaling by concentrating updates where they matter most. When the environment changes, model accuracy cannot be assumed, and a staleness mechanism like Dyna-Q+'s exploration bonus is necessary to detect and correct stale beliefs.

## Section 3: AI Use Reflection

### 3.1 Initial Interaction

I began by specifying roles, asking Claude to act as tutor and critic; I would design the implementation and draft the report. Our first task was to clarify the distinction between model-free and model-based RL, which Claude probed with a thought experiment about what happens after one real step with 50 planning iterations. This discussion ensured that I understood the role the model plays in developing policy and supported the Section One draft.

### 3.2 Iteration Cycle

#### 3.2.1 Iteration Cycle - Example #1

In drafting the project overview, I wrote that "Q-planning will develop policies multiple times per step." Claude challenged the phrasing on two counts: planning doesn't develop policy, and Q-planning isn't standard terminology.  I shared a citation from the textbook that supported the role planning plays in developing policy. Claude conceded that it was valid, so I kept the policy-development framing, supported by the citation, but accepted the terminology critique by dropping 'Q-planning.'

#### 3.2.2 Iteration Cycle - Example #2

A lengthy exchange with Claude led me to understand that there are four key data structures that enable model-based RL with prioritized sweeping: the Q-table, model, reverse index, and priority queue. Along the way I corrected several misconceptions: I had typed the queue as a stack, and I thought it was ordered by proximity rather than absolute TD error. I synthesized the four-structure summary myself.

#### 3.2.3 Iteration Cycle - Example #3

When I was reviewing Gymnasium documentation, I saw that Taxi-v3 had been deprecated. I proceeded to plan the implementation around Taxi-v4. When Claude ran the first iteration's tests, it reported a VersionNotFound error because my conda environment was configured with an earlier version of gymnasium. I upgraded gymnasium and our tests went green.

### 3.3 Critical Evaluation

Iterative development with a steel-thread proof-of-concept exposed an early Claude over-engineering: the per-step trace schema produced 12 MB files after one short Q-learning run. I proposed per-episode aggregation as an alternative, which preserved every metric the visualizations needed at ~370 KB per run. Test-first refactoring and re-running the cumulative-reward demo confirmed the new schema produced the same learning curve before we committed.

### 3.4 Learning Reflection

Debugging the prioritized sweeping implementation revealed a key misconception: I understood that the algorithm traverses predecessor states, but assumed proximity determined which predecessors mattered. Inspecting the TD error formula clarified that priority is determined by how wrong a predecessor's Q-value is, not where it is. Working with AI, I learned that steel-thread MVPs surface gaps between intent and implementation early, before rework becomes expensive. Going forward, I will continue using CLAUDE.md to establish and persist collaboration guidelines from the start.

## Section 4: Speaker Notes

- **Problem:** Pure Q-learning is sample-inefficient. Planning-based methods exploit a learned model to perform multiple Q-updates per real step, compressing the learning timeline. Uniform random planning is inefficient; prioritized sweeping addresses that by focusing on updates with the highest expected TD error.

- **Method:** Four tabular agents in a class hierarchy — Q-learning baseline, Dyna-Q with uniform random planning, Dyna-Q+ with a staleness exploration bonus, and prioritized sweeping with a TD-error priority queue and reverse model index.

- **Design decision:** Per-episode trace aggregation instead of per-step recording reduced file sizes from ~12 MB to ~370 KB per run without losing any metric needed for the visualizations or trade-off analysis.

- **Key result:** Dyna-Q (n=50) converged 5.5x faster than Q-learning in real steps. Prioritized sweeping matched that sample efficiency at one-seventh the wall time, by directing planning updates toward the highest-TD-error transitions.

- **Challenge:** Claude Code can overengineer solutions in ways that are not immediately obvious. The per-step trace schema — 12 MB files, far more data than the visualizations needed — went undetected until a steel-thread run made the cost visible. Small increments and early inspection are necessary to catch this.

- **Insight:** Model-based planning is most valuable when real interactions are expensive and the model is accurate. When the environment is non-stationary, model accuracy cannot be assumed, and an explicit staleness mechanism is necessary to detect and correct stale beliefs.